### NER (Named Entity Recognition)

In [1]:
import spacy 

nlp = spacy.load('en_core_web_sm')

sentence = 'Why Apple is looking at buying U.K. startup for $1 billion ?'

doc = nlp(sentence)

for ent in doc.ents:
    print(ent.text, ent.start_char, ent.end_char, ent.label_)

Apple 4 9 ORG
U.K. 31 35 GPE
$1 billion 48 58 MONEY


In [2]:
# Case Sensitive
sentence = 'Why apple is looking at buying U.K. startup for $1 billion ?'

doc = nlp(sentence)

for ent in doc.ents:
    print(ent.text, ent.label_)

U.K. GPE
$1 billion MONEY


In [ ]:

df = pd.read_csv('Datasets/book_reviews_sample.csv')
df.head()

KeyboardInterrupt: 

In [6]:
df = df.rename(columns={
    "reviewText": "Reviews",
    "rating": "Ratings"
})

In [7]:
df.head()

,index,Reviews,Ratings
0,11494,Clean and funny. A bit busy with all the diffe...,3
1,984,Alex a sexy hot cop and the PhD candidate. Wha...,4
2,1463,Good thing that this is a free story. I read i...,1
3,10342,"Action, action, action! Equipment keeps gettin...",4
4,5256,this was hands down the worse book i have ever...,1


In [ ]:
df[['Reviews', 'Ratings']].to_csv('book_reviews.csv', index=False)



,Reviews,Ratings
0,Clean and funny. A bit busy with all the diffe...,3
1,Alex a sexy hot cop and the PhD candidate. Wha...,4
2,Good thing that this is a free story. I read i...,1
3,"Action, action, action! Equipment keeps gettin...",4
4,this was hands down the worse book i have ever...,1


In [6]:
import pandas as pd

df = pd.read_csv('book_reviews.csv')
df.head()

AttributeError: partially initialized module 'pandas' has no attribute 'core' (most likely due to a circular import)

## 1. NER with Spacy

In [13]:
import spacy

nlp = spacy.load('en_core_web_sm')

for text in df['Reviews'][:20]:
    doc = nlp(text)
    for ent in doc.ents:
        print(f'{ent.text} --> {ent.label_}')

Alex --> PERSON
PhD --> WORK_OF_ART
a few years ago --> DATE
Kinkle --> GPE
Dan --> PERSON
Elle --> PERSON
1 --> CARDINAL


## 2. Sentimental Analysis

In [14]:
# 2.1 TextBlob 
from textblob import TextBlob
text_blob_scores = []

for text in df['Reviews']:
    score = TextBlob(text).sentiment.polarity
    text_blob_scores.append(score)
print(text_blob_scores[:10])

[0.23611111111111108, 0.43, 0.18750000000000003, 0.11534090909090909, -0.2777777777777778, 0.3333333333333333, 0.38, 0.6666666666666666, 0.0, 0.6]


In [16]:
# 2.2 VADER
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader_scores = []

for text in df['Reviews']:
    score = SentimentIntensityAnalyzer().polarity_scores(text)
    print(score)
    vader_scores.append(score['compound'])


{'neg': 0.0, 'neu': 0.69, 'pos': 0.31, 'compound': 0.7684}
{'neg': 0.0, 'neu': 0.548, 'pos': 0.452, 'compound': 0.9325}
{'neg': 0.062, 'neu': 0.707, 'pos': 0.231, 'compound': 0.674}
{'neg': 0.0, 'neu': 0.747, 'pos': 0.253, 'compound': 0.6948}
{'neg': 0.162, 'neu': 0.838, 'pos': 0.0, 'compound': -0.4767}
{'neg': 0.193, 'neu': 0.433, 'pos': 0.373, 'compound': 0.75}
{'neg': 0.0, 'neu': 0.858, 'pos': 0.142, 'compound': 0.5106}
{'neg': 0.0, 'neu': 0.731, 'pos': 0.269, 'compound': 0.7906}
{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
{'neg': 0.0, 'neu': 0.721, 'pos': 0.279, 'compound': 0.8107}
{'neg': 0.0, 'neu': 0.807, 'pos': 0.193, 'compound': 0.5106}
{'neg': 0.198, 'neu': 0.802, 'pos': 0.0, 'compound': -0.5334}
{'neg': 0.084, 'neu': 0.793, 'pos': 0.123, 'compound': 0.2477}
{'neg': 0.0, 'neu': 0.728, 'pos': 0.272, 'compound': 0.7745}
{'neg': 0.136, 'neu': 0.864, 'pos': 0.0, 'compound': -0.4585}
{'neg': 0.323, 'neu': 0.509, 'pos': 0.167, 'compound': -0.4838}
{'neg': 0.091, 'neu': 0.

In [17]:
bins = [-1, -0.1, 0.1, 1]
names = ['Negative', 'Neutral', 'Positive']
df['textblob_labels'] = pd.cut(text_blob_scores, bins, labels=names)
df['vader_labels'] = pd.cut(vader_scores, bins, labels=names)

In [18]:
df.head()

,Reviews,Ratings,textblob_labels,vader_labels
0,Clean and funny. A bit busy with all the diffe...,3,Positive,Positive
1,Alex a sexy hot cop and the PhD candidate. Wha...,4,Positive,Positive
2,Good thing that this is a free story. I read i...,1,Positive,Positive
3,"Action, action, action! Equipment keeps gettin...",4,Positive,Positive
4,this was hands down the worse book i have ever...,1,Negative,Negative


## 3. Transformer Models - BERT

In [3]:
!pip3 install torch transformers accelerate datasets scikit-learn pandas numpy


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 73.6 MB 2.8 MB/s eta 0:00:01     |███████████████████████▌        | 54.1 MB 2.4 MB/s eta 0:00:09
     |████████████████████████████████| 12.0 MB 2.3 MB/s eta 0:00:01
     |████████████████████████████████| 374 kB 2.3 MB/s eta 0:00:01
     |████████████████████████████████| 512 kB 2.3 MB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 2.1 MB/s eta 0:00:01
     |████████████████████████████████| 10.8 MB 328 kB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 3.2 MB/s eta 0:00:01
     |████████████████████████████████| 6.3 MB 4.4 MB/s eta 0:00:01�███        | 4.7 MB 4.4 MB/s eta 0:00:01
     |████████████████████████████████| 134 kB 2.5 MB/s eta 0:00:01
     |████████████████████████████████| 1.6 MB 2.5 MB/s eta 0:00:01
     |████████████████████████████████| 200 kB 3.0 MB/s eta 0:00:01
     |████████████████████████████████| 288 kB 3.1 MB/s eta 0

In [20]:
df.head()

,Reviews,Ratings,textblob_labels,vader_labels
0,Clean and funny. A bit busy with all the diffe...,3,Positive,Positive
1,Alex a sexy hot cop and the PhD candidate. Wha...,4,Positive,Positive
2,Good thing that this is a free story. I read i...,1,Positive,Positive
3,"Action, action, action! Equipment keeps gettin...",4,Positive,Positive
4,this was hands down the worse book i have ever...,1,Negative,Negative


In [22]:
def map_sentiment(rating):
    if rating <=2:
        return 0
    elif rating ==3:
        return 1
    else:
        return 2
    
df['label'] = df['Ratings'].apply(map_sentiment)
df.head()

,Reviews,Ratings,textblob_labels,vader_labels,label
0,Clean and funny. A bit busy with all the diffe...,3,Positive,Positive,1
1,Alex a sexy hot cop and the PhD candidate. Wha...,4,Positive,Positive,2
2,Good thing that this is a free story. I read i...,1,Positive,Positive,0
3,"Action, action, action! Equipment keeps gettin...",4,Positive,Positive,2
4,this was hands down the worse book i have ever...,1,Negative,Negative,0


In [23]:
from transformers import BertTokenizer 

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# tokenize text 
def tokenize(batch):
    return tokenizer(batch['Reviews'], padding='max_length', truncation=True, max_length=128)

In [ ]:
from datasets import Dataset 

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize, batched=True)



Map: 100%|██████████| 100/100 [00:00<00:00, 2140.65 examples/s]


Dataset({
    features: ['Reviews', 'Ratings', 'textblob_labels', 'vader_labels', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 100
})

In [26]:
dataset = dataset.remove_columns(['Reviews', 'Ratings', 'textblob_labels', 'vader_labels'])
dataset

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 100
})

In [27]:
dataset.set_format('torch')

In [28]:
dataset = dataset.train_test_split(test_size=0.2)
train_ds= dataset['train']
test_ds = dataset['test']

In [29]:
from transformers import BertForSequenceClassification 

model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [35]:
!pip install 'accelerate>=0.26.0'

In [39]:
!/Users/orbinsunny/.pyenv/versions/3.12.3/bin/python -m pip install --upgrade --force-reinstall accelerate

  Using cached accelerate-1.12.0-py3-none-any.whl.metadata (19 kB)
  Using cached numpy-2.4.0-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached psutil-7.2.1-cp36-abi3-macosx_11_0_arm64.whl.metadata (22 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (2.4 kB)
  Using cached torch-2.9.1-cp312-none-macosx_11_0_arm64.whl.metadata (30 kB)
  Using cached safetensors-0.7.0-cp38-abi3-macosx_11_0_arm64.whl.metadata (4.1 kB)
  Using cached filelock-3.20.2-py3-none-any.whl.metadata (2.1 kB)
  Using cached fsspec-2025.12.0-py3-none-any.whl.metadata (10 kB)
  Using cached hf_xet-1.2.0-cp37-abi3-macosx_11_0_arm64.whl.metadata (4.9 kB)
  Using cached httpx-0.28.1-py3-none-any.whl.metadata (7.1 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached certifi-2026.1.4-py3-none-any.whl.metadata (2.5 

In [40]:
import accelerate
print(accelerate.__version__)

1.12.0


In [1]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./bert_sentimnet',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    weight_decay=0.01
)

ModuleNotFoundError: No module named 'transformers'

## 4. GPT

## 5. Sequence Models - LSTM

## 6. Translation 

Rest of the code

https://colab.research.google.com/drive/1Ko2krNyfl1aGSPQW5bxt9RXVQyfb6Psp